In [1]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import direction_utils as utils
import direction_learning_utils as train_utils


In [2]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=32):
        super(SEBlock, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)  # Output size: (batch_size, in_channels, 1, 1)
        
        # self.fc1 = nn.Linear(in_channels, max(1, in_channels // reduction), bias=False)  # Squeeze
        self.fc1 = nn.Linear(in_channels, in_channels // reduction, bias=False)  # Squeeze

        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels, bias=False)  # Excitation
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, channels, height, width = x.size()
        
        # Squeeze: Global Average Pooling
        out = self.global_avg_pool(x).view(batch_size, channels)  # Shape: (batch_size, in_channels)
        
        # Excitation: Fully connected layers
        out = self.fc1(out)  # Shape: (batch_size, in_channels // reduction)
        out = self.relu(out)
        out = self.fc2(out)  # Shape: (batch_size, in_channels)
        out = self.sigmoid(out).view(batch_size, channels, 1, 1)  # Reshape to (batch_size, in_channels, 1, 1)
        
        # Scale the input by the SE weights
        return x * out.expand_as(x)

In [3]:
class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=27, Samples=2500, dropoutRate=0.5, 
                 kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
        super(EEGNet, self).__init__()
        
        # Handle dropout type
        if dropoutType == 'SpatialDropout2D':
            self.dropout = nn.Dropout2d(dropoutRate)
        elif dropoutType == 'Dropout':
            self.dropout = nn.Dropout(dropoutRate)
        else:
            raise ValueError('dropoutType must be one of SpatialDropout2D or Dropout.')

        # Block 1
        self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
        self.batchnorm1 = nn.BatchNorm2d(F1)
        # Squeeze-and-Excitation Block
        self.se1 = SEBlock(F1, 2)


        self.depthwiseConv = nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(F1*D)
        self.se2 = SEBlock(F1*D, 3)  # Squeeze-and-Excitation Block
        self.pool1 = nn.AvgPool2d((1, 4))

        # Block 2
        self.separableConv = nn.Conv2d(F1*D, F2, (1, 16), padding='same', bias=False)
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.se3 = SEBlock(F2, 3)  # Squeeze-and-Excitation Block
        self.pool2 = nn.AvgPool2d((1, 8))

        # Flatten and Dense
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(F2 * (Samples // (4 * 8)), nb_classes)
        self.norm_constraint = nn.utils.weight_norm(self.dense)

    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.se1(x)
        x = self.depthwiseConv(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.se2(x)
        x = self.pool1(x)
        x = self.dropout(x)

        # Block 2
        x = self.separableConv(x)
        x = self.batchnorm3(x)
        x = F.elu(x)
        x = self.se3(x)
        x = self.pool2(x)
        x = self.dropout(x)

        # Flatten and Dense
        x = self.flatten(x)
        x = self.dense(x)
        return F.softmax(x, dim=1)

In [4]:
def model_using_calibration_data():
    torch.manual_seed(0)
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    batch_size=32
    fs = 500
    train_ratio = 0.9
    sub = 7 #Till subject 7 (starting from 0), only calibration sessions are conducted

    # Xtr, Ytr = create_dataset(sub, base_path=parent_dir)
    Xtr, Ytr = utils.calib_sess_dataset(base_path=parent_dir)
    X_train = utils.baseline_correction(Xtr)
    X_train = utils.bandpass_filtering(X_train)

    # Creating train-validation split
    eeg_train, eeg_val, label_train, label_val = train_test_split(X_train, Ytr, 
                                                                train_size=train_ratio, random_state=42, shuffle=True)


    X_train_tensor = torch.tensor(eeg_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_train_tensor = torch.tensor(label_train, dtype=torch.long).to(device)  # Use long for classification

    X_val_tensor = torch.tensor(eeg_val, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_val_tensor = torch.tensor(label_val, dtype=torch.long).to(device)

    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)


    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    # optimizer = optim.Adam(model.parameters(), lr=1e-3)
    trained_model = train_utils.model_training(model, train_loader, val_loader)
    torch.save(trained_model.state_dict(), 'calibrated_eegnetSE_model.pth')  # Save best model

    return trained_model
    

In [5]:
def evaluation_on_online_data(model_file):
    torch.manual_seed(0)
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size=32

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model.load_state_dict(torch.load(f'{model_file}.pth'))

    online_perf = dict()
    for sub in range(8, 21):
        Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)

        test_loader = train_utils.convert_to_tensor(Xte, Yte)

        test_loss, test_acc = train_utils.model_evaluation(model, test_loader)
        online_perf[f'Sub{sub:02d}'] = test_acc
        
        print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

    print(f'Average Accuracy: {np.mean(list(online_perf.values()))}')
    print(list(online_perf.values()))

    return
    

In [6]:
# Model Training on Calibration and Testing on Online Session Data
model_using_calibration_data()
evaluation_on_online_data('calibrated_eegnetSE_model')

d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\modules\conv.py:454: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv2d(input, weight, bias, self.stride,
d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
C:\Users\postd\AppData\Local\Temp\ipykernel_21020\359627777.py:8: FutureWarning: You are using `torch.load` with `weights_only=False`

Subject: 08, Test Accuracy: 59.38%
Subject: 09, Test Accuracy: 53.12%
Subject: 10, Test Accuracy: 51.56%
Subject: 11, Test Accuracy: 48.44%
Subject: 12, Test Accuracy: 48.44%
Subject: 13, Test Accuracy: 57.81%
Subject: 14, Test Accuracy: 67.19%
Subject: 15, Test Accuracy: 53.12%
Subject: 16, Test Accuracy: 53.12%
Subject: 17, Test Accuracy: 64.06%
Subject: 18, Test Accuracy: 43.75%
Subject: 19, Test Accuracy: 45.31%
Subject: 20, Test Accuracy: 64.06%
Average Accuracy: 54.56730769230769
[59.375, 53.125, 51.5625, 48.4375, 48.4375, 57.8125, 67.1875, 53.125, 53.125, 64.0625, 43.75, 45.3125, 64.0625]


In [16]:
# Model Fine Tuning
def model_fine_tuning_params(model_file, denseLayer=True, conv2dLayer=False):

    torch.manual_seed(0)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model.load_state_dict(torch.load(f'{model_file}.pth'))

    for param in model.parameters():
            param.requires_grad = False

    # Unfreeze the last layer parameters
    for param in model.se1.parameters():
        param.requires_grad = True  # Unfreeze last layer
    
    # Unfreeze the last layer parameters
    for param in model.se2.parameters():
        param.requires_grad = True  # Unfreeze last layer
    
    # Unfreeze the last layer parameters
    for param in model.se3.parameters():
        param.requires_grad = True  # Unfreeze last layer


    if denseLayer or conv2dLayer:

        # for param in model.parameters():
        #     param.requires_grad = False

        # # Unfreeze the last layer parameters
        # for param in model.se1.parameters():
        #     param.requires_grad = True  # Unfreeze last layer
        
        # # Unfreeze the last layer parameters
        # for param in model.se2.parameters():
        #     param.requires_grad = True  # Unfreeze last layer
        
        # # Unfreeze the last layer parameters
        # for param in model.se3.parameters():
        #     param.requires_grad = True  # Unfreeze last layer

        if denseLayer:
            print(f'Dense layer is fine-tuned')
            # Freeze all layers except the last one
            for param in model.dense.parameters():
                param.requires_grad = True  # Freeze all parameters

        if conv2dLayer:
            print(f'Conv2D layer is fine-tuned')
            # Unfreeze the last layer parameters
            for param in model.separableConv.parameters():
                param.requires_grad = True  # Unfreeze last layer

    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    return model, optimizer


In [17]:
def subject_specific_fine_tuning(denseLayer=True, conv2dLayer=False):
    parent_dir = os.path.dirname(os.getcwd())
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    Xcalib, Ycalib = utils.calib_sess_dataset(base_path=parent_dir)

    train_ratio = 0.9
    batch_size=32
    online_perf = dict()

    for sub in range(8, 21):
        Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)
        eeg_train, eeg_val, label_train, label_val = train_test_split(Xtr, Ytr, 
                                                                train_size=train_ratio, random_state=42, shuffle=True)
        
        # Xtrain = np.concatenate((Xcalib, eeg_train), axis=0)
        # Ytrain = np.concatenate((Ycalib, label_train), axis=0)
        Xtrain = eeg_train
        Ytrain = label_train
        
        train_loader = train_utils.convert_to_tensor(Xtrain, Ytrain)
        val_loader = train_utils.convert_to_tensor(eeg_val, label_val)

        model, optimizer = model_fine_tuning_params('calibrated_eegnetSE_model', denseLayer=denseLayer, conv2dLayer=conv2dLayer)
        trained_model = train_utils.model_training(model, train_loader, val_loader)

        test_loader = train_utils.convert_to_tensor(Xte, Yte)

        # model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
        # model.load_state_dict(torch.load('trained_model_checkpoint.pth'))

        test_loss, test_acc = train_utils.model_evaluation(trained_model, test_loader)

        print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')
        online_perf[f'Sub{sub:02d}'] = test_acc

    print(f'Average Accuracy: {np.mean(list(online_perf.values()))}')
    print(list(online_perf.values()))


In [18]:
subject_specific_fine_tuning(denseLayer=False, conv2dLayer=False)

C:\Users\postd\AppData\Local\Temp\ipykernel_21020\1501784121.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_file}.pth'))


Subject: 08, Test Accuracy: 56.25%
Subject: 09, Test Accuracy: 60.94%
Subject: 10, Test Accuracy: 45.31%
Subject: 11, Test Accuracy: 48.44%
Subject: 12, Test Accuracy: 53.12%
Subject: 13, Test Accuracy: 71.88%
Subject: 14, Test Accuracy: 60.94%
Subject: 15, Test Accuracy: 53.12%
Subject: 16, Test Accuracy: 59.38%
Subject: 17, Test Accuracy: 45.31%
Subject: 18, Test Accuracy: 43.75%
Subject: 19, Test Accuracy: 59.38%
Subject: 20, Test Accuracy: 70.31%
Average Accuracy: 56.00961538461539
[56.25, 60.9375, 45.3125, 48.4375, 53.125, 71.875, 60.9375, 53.125, 59.375, 45.3125, 43.75, 59.375, 70.3125]


In [14]:
subject_specific_fine_tuning(denseLayer=True, conv2dLayer=False)

C:\Users\postd\AppData\Local\Temp\ipykernel_21020\3978564272.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_file}.pth'))


Dense layer is fine-tuned
Subject: 08, Test Accuracy: 46.88%
Dense layer is fine-tuned
Subject: 09, Test Accuracy: 53.12%
Dense layer is fine-tuned
Subject: 10, Test Accuracy: 54.69%
Dense layer is fine-tuned
Subject: 11, Test Accuracy: 54.69%
Dense layer is fine-tuned
Subject: 12, Test Accuracy: 64.06%
Dense layer is fine-tuned
Subject: 13, Test Accuracy: 54.69%
Dense layer is fine-tuned
Subject: 14, Test Accuracy: 42.19%
Dense layer is fine-tuned
Subject: 15, Test Accuracy: 54.69%
Dense layer is fine-tuned
Subject: 16, Test Accuracy: 45.31%
Dense layer is fine-tuned
Subject: 17, Test Accuracy: 51.56%
Dense layer is fine-tuned
Subject: 18, Test Accuracy: 57.81%
Dense layer is fine-tuned
Subject: 19, Test Accuracy: 65.62%
Dense layer is fine-tuned
Subject: 20, Test Accuracy: 75.00%
Average Accuracy: 55.40865384615385
[46.875, 53.125, 54.6875, 54.6875, 64.0625, 54.6875, 42.1875, 54.6875, 45.3125, 51.5625, 57.8125, 65.625, 75.0]


In [15]:
subject_specific_fine_tuning(denseLayer=True, conv2dLayer=True)

C:\Users\postd\AppData\Local\Temp\ipykernel_21020\3978564272.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f'{model_file}.pth'))


Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 08, Test Accuracy: 50.00%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 09, Test Accuracy: 56.25%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 10, Test Accuracy: 60.94%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 11, Test Accuracy: 53.12%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 12, Test Accuracy: 68.75%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 13, Test Accuracy: 60.94%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 14, Test Accuracy: 53.12%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 15, Test Accuracy: 45.31%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 16, Test Accuracy: 51.56%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 17, Test Accuracy: 50.00%
Dense layer is fine-tuned
Conv2D layer is fine-tuned
Subject: 18, Test Accuracy: 57.81%
Dense layer is fine-tuned
Conv2D